In [2]:
import pandas as pd

df = pd.read_parquet("hf://datasets/biglam/gutenberg-poetry-corpus/data/train-00000-of-00001-fa9fb9e1f16eed7e.parquet")

d:\Facultate\Inteligenta artificiala\Laborator\Laborator 9\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df.head()

,line,gutenberg_id
0,The Song of Hiawatha is based on the legends a...,19
1,"many North American Indian tribes, but especia...",19
2,"Ojibway Indians of northern Michigan, Wisconsi...",19
3,"They were collected by Henry Rowe Schoolcraft,...",19
4,"Schoolcraft married Jane, O-bah-bahm-wawa-ge-z...",19


In [4]:
poems = df.groupby('gutenberg_id')['line'].apply(lambda x: ' \n '.join(x)).reset_index()
poems.rename(columns={'line': 'text'}, inplace=True)

print("Processed poems dataframe:")
print(poems.head())

Processed poems dataframe:
   gutenberg_id                                               text
0            19  The Song of Hiawatha is based on the legends a...
1            20  Of that Forbidden Tree, whose mortal tast \n B...
2            26  Of Man's first disobedience, and the fruit \n ...
3            58  I, WHO erewhile the happy Garden sung \n By on...
4           109  All I could see from where I stood \n The room...


In [5]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6702.63it/s]


In [6]:
import re

model.config.pad_token_id = model.config.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

def generate_poem(prompt, max_len, temp, k, p):

    inputs = tokenizer(prompt, return_tensors='pt')
    input_ids = inputs['input_ids']
    
    outputs = model.generate(
        input_ids,
        max_length=max_len,
        num_return_sequences=1,
        do_sample=True,
        temperature=temp,
        top_k=k,
        top_p=p,
        pad_token_id=tokenizer.eos_token_id
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def get_first_lines_from_stanzas(poem_text):
    stanzas = re.split(r'\s*\n\s*\n\s*', poem_text.strip())
    
    first_lines = [s.strip().split('\n')[0] for s in stanzas if s.strip()]
    
    return first_lines

original_poem = poems['text'].iloc[2] 
print("--- POEZIA ORIGINALĂ ---")

first_lines = get_first_lines_from_stanzas(original_poem)
prompt = first_lines[0]
print(prompt)

max_len = 60

--- POEZIA ORIGINALĂ ---
Of Man's first disobedience, and the fruit 


In [7]:
# --- 1. Variația Temperaturii (Temperature) ---
# Temperature controlează caracterul aleatoriu. Valori mai mici = mai previzibil.
print("--- EXPERIMENTE CU TEMPERATURE (top_k=50, top_p=0.95) ---")

# Temp scăzută: text mai coerent, dar potențial repetitiv
temp_low = 0.7
print(f"\n[Temp = {temp_low}]")
print(generate_poem(prompt, max_len, temp=temp_low, k=50, p=0.95))

# Temp medie: un echilibru între coerență și creativitate
temp_medium = 1.0
print(f"\n[Temp = {temp_medium}]")
print(generate_poem(prompt, max_len, temp=temp_medium, k=50, p=0.95))

# Temp ridicată: text foarte creativ, dar cu risc de a deveni incoerent
temp_high = 1.5
print(f"\n[Temp = {temp_high}]")
print(generate_poem(prompt, max_len, temp=temp_high, k=50, p=0.95))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- EXPERIMENTE CU TEMPERATURE (top_k=50, top_p=0.95) ---

[Temp = 0.7]
Of Man's first disobedience, and the fruit  of his disobedience, is the work of the Spirit. The Spirit, therefore, has placed a stop to this work, and has not stopped it. This is because in the Spirit, there are two things. The first is the work of the

[Temp = 1.0]
Of Man's first disobedience, and the fruit ichor of that obedience, to wit: and he gave the name of the good unto the evil (which he bore as an example, and commanded that no one should be guilty, that is, had no right of any evil deed unto him,

[Temp = 1.5]
Of Man's first disobedience, and the fruit ˜The Law of Life (1601)--It was the one day of my Lord Jesus (the resurrection of his Father). 2 At this, a servant who had his hair fastened by ten men gathered it, and gave me an ear for which


In [8]:
# --- 2. Variația Top-K ---
# Top-K limitează vocabularul la cele 'k' cele mai probabile cuvinte.
print("\n\n--- EXPERIMENTE CU TOP_K (temp=1.0, top_p=0.95) ---")

# K mic: limitează drastic alegerile, text mai sigur
k_low = 10
print(f"\n[Top_K = {k_low}]")
print(generate_poem(prompt, max_len, temp=1.0, k=k_low, p=0.95))

# K mare: permite mai multă varietate
k_high = 100
print(f"\n[Top_K = {k_high}]")
print(generate_poem(prompt, max_len, temp=1.0, k=k_high, p=0.95))



--- EXPERIMENTE CU TOP_K (temp=1.0, top_p=0.95) ---

[Top_K = 10]
Of Man's first disobedience, and the fruit iced in his mouth. He was not so much a prisoner of the Roman Church, but a slave to the Roman Empire, and was the first to be converted to the faith of the Church. He became the first Christian to be baptized, and,

[Top_K = 100]
Of Man's first disobedience, and the fruit  of his great rebellion, has been made over in one little square.
The beginning of The Red Man (or whatever the name of The Red Man is) is as follows.   In The Red Man he has found a means of keeping the


In [9]:
# --- 3. Variația Top-P (Nucleus Sampling) ---
# Top-P selectează dintr-un set de cuvinte a căror probabilitate cumulată depășește 'p'.
print("\n\n--- EXPERIMENTE CU TOP_P (temp=1.0, top_k=0) ---")
# Setăm top_k=0 pentru a folosi doar top_p

# P mic: set de cuvinte mai restrâns, text mai concentrat
p_low = 0.80
print(f"\n[Top_P = {p_low}]")
print(generate_poem(prompt, max_len, temp=1.0, k=0, p=p_low))

# P mare: set de cuvinte mai larg și dinamic, mai multă creativitate
p_high = 0.95
print(f"\n[Top_P = {p_high}]")
print(generate_poem(prompt, max_len, temp=1.0, k=0, p=p_high))



--- EXPERIMENTE CU TOP_P (temp=1.0, top_k=0) ---

[Top_P = 0.8]
Of Man's first disobedience, and the fruit  of his repentance is a seal to his enemies, and a gift of pardon unto them who will enter the house of David. 12. He will set the walls at the entrance to their house and entice his people to go out and gather together

[Top_P = 0.95]
Of Man's first disobedience, and the fruit  of that disobedience was the dream of many lives. Helping others can be hard work, but for those that do help others, their heart goes out to them and is to be opened in their body, their spirit to be subdued in every movement


In [10]:
import torch
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

In [11]:
try:
    poems_subset = poems

    dataset = Dataset.from_pandas(poems_subset[['text']])

    def tokenize_function(examples):
        # Tokenizăm fiecare poezie. `truncation=True` asigură că poeziile foarte lungi sunt tăiate.
        return tokenizer(examples["text"], truncation=True, max_length=512)

    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    
    tokenizer.pad_token = tokenizer.eos_token
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    model_name = 'gpt2'
    model = GPT2LMHeadModel.from_pretrained(model_name)

    training_args = TrainingArguments(
        output_dir="./trained-model-finetuned",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        save_steps=10_000,
        save_total_limit=2,
        logging_steps=500,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=tokenized_dataset,
    )

    print("--- Începe procesul de fine-tuning... ---")
    trainer.train()
    print("--- Fine-tuning finalizat! ---")

    fine_tuned_model_path = "./trained-model-finetuned"
    trainer.save_model(fine_tuned_model_path)
    tokenizer.save_pretrained(fine_tuned_model_path)
    print(f"Modelul adaptat a fost salvat în: {fine_tuned_model_path}")

except Exception as e:
    print(f"A apărut o eroare în timpul procesului de fine-tuning: {e}")


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8647.45it/s]


--- Începe procesul de fine-tuning... ---


d:\Facultate\Inteligenta artificiala\Laborator\Laborator 9\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,3.815079


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]


--- Fine-tuning finalizat! ---


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]

Modelul adaptat a fost salvat în: ./trained-model-finetuned


In [12]:
fine_tuned_model_path = "./trained-model-finetuned"

tokenizer = GPT2Tokenizer.from_pretrained(fine_tuned_model_path)
model = GPT2LMHeadModel.from_pretrained(fine_tuned_model_path)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7979.80it/s]


In [16]:
def generate_poem_ft(prompt, max_len, temp, k, p):

    inputs = tokenizer(prompt, return_tensors='pt')
    input_ids = inputs['input_ids']
    
    outputs = model.generate(
        input_ids,
        max_length=max_len,
        num_return_sequences=1,
        do_sample=True,
        temperature=temp,
        top_k=k,
        top_p=p,
        pad_token_id=tokenizer.eos_token_id
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

original_poem = poems['text'].iloc[2] 
print("--- POEZIA ORIGINALĂ ---")

first_lines = get_first_lines_from_stanzas(original_poem)
prompt = first_lines[0]
print(prompt)

max_len = 60

--- POEZIA ORIGINALĂ ---
Of Man's first disobedience, and the fruit 


In [14]:
# --- 1. Variația Temperaturii (Temperature) ---
# Temperature controlează caracterul aleatoriu. Valori mai mici = mai previzibil.
print("--- EXPERIMENTE CU TEMPERATURE (top_k=50, top_p=0.95) ---")

# Temp scăzută: text mai coerent, dar potențial repetitiv
temp_low = 0.7
print(f"\n[Temp = {temp_low}]")
print(generate_poem(prompt, max_len, temp=temp_low, k=50, p=0.95))

# Temp medie: un echilibru între coerență și creativitate
temp_medium = 1.0
print(f"\n[Temp = {temp_medium}]")
print(generate_poem(prompt, max_len, temp=temp_medium, k=50, p=0.95))

# Temp ridicată: text foarte creativ, dar cu risc de a deveni incoerent
temp_high = 1.5
print(f"\n[Temp = {temp_high}]")
print(generate_poem(prompt, max_len, temp=temp_high, k=50, p=0.95))

--- EXPERIMENTE CU TEMPERATURE (top_k=50, top_p=0.95) ---

[Temp = 0.7]
Of Man's first disobedience, and the fruit 
 Of the eternal life: 
 I have to be his slave! 
 I have to live in my own house, 
 And, if you like, 
 You can call me my slave!" 
 "Oh, my boy

[Temp = 1.0]
Of Man's first disobedience, and the fruit 
 That shall rise up and die, 
 I, or one from a mother's womb, 
 And give the world my name, 
 Till I have finished them all.
 "O my Lord, the only reward I can give

[Temp = 1.5]
Of Man's first disobedience, and the fruit 
 Of God's first-taught love, 
 Shall end with salvation, shall seem 
 To be eternal, 
 The last of these things 
 So we do strive the light upon ours: and at this 
 end shall


In [17]:
# --- 2. Variația Top-K ---
# Top-K limitează vocabularul la cele 'k' cele mai probabile cuvinte.
print("\n\n--- EXPERIMENTE CU TOP_K (temp=1.0, top_p=0.95) ---")

# K mic: limitează drastic alegerile, text mai sigur
k_low = 10
print(f"\n[Top_K = {k_low}]")
print(generate_poem(prompt, max_len, temp=1.0, k=k_low, p=0.95))

# K mare: permite mai multă varietate
k_high = 100
print(f"\n[Top_K = {k_high}]")
print(generate_poem(prompt, max_len, temp=1.0, k=k_high, p=0.95))



--- EXPERIMENTE CU TOP_K (temp=1.0, top_p=0.95) ---

[Top_K = 10]
Of Man's first disobedience, and the fruit 
 Of his first labor, and of his first labor 
 In the same manner as that which we have already seen, 
 The man who, like a man, has not yet been 
 In labor, but yet was able to earn

[Top_K = 100]
Of Man's first disobedience, and the fruit 
 Of his first disobedience and the fruit 
 Of his first disobedience, 
 _The_ 
 _In that state, he first sent, 
 _The_ 
 _Here, in the second generation, he sent, 


In [18]:
# --- 3. Variația Top-P (Nucleus Sampling) ---
# Top-P selectează dintr-un set de cuvinte a căror probabilitate cumulată depășește 'p'.
print("\n\n--- EXPERIMENTE CU TOP_P (temp=1.0, top_k=0) ---")
# Setăm top_k=0 pentru a folosi doar top_p

# P mic: set de cuvinte mai restrâns, text mai concentrat
p_low = 0.80
print(f"\n[Top_P = {p_low}]")
print(generate_poem(prompt, max_len, temp=1.0, k=0, p=p_low))

# P mare: set de cuvinte mai larg și dinamic, mai multă creativitate
p_high = 0.95
print(f"\n[Top_P = {p_high}]")
print(generate_poem(prompt, max_len, temp=1.0, k=0, p=p_high))



--- EXPERIMENTE CU TOP_P (temp=1.0, top_k=0) ---

[Top_P = 0.8]
Of Man's first disobedience, and the fruit 
 Of Man's first disobedience. (A.D. 1876.) ** 
 All world these days, in all 
 In the world we live. 
 God sends us to help; 
 But we must make a new effort

[Top_P = 0.95]
Of Man's first disobedience, and the fruit 
 Which makes him glad to see it ever with me. 
 Did you see me in my mother's womb 
 On bosom short-falling flowers, 
 And skinned white as a baby? 
 The breezes shri


### c.1 Care sunt diferențele de calitate între textele generate cu cele două tipuri de LLM-uri?

Cele două LLM-uri pe care le-am folosit sunt:
1.  **LLM Pre-antrenat (GPT-2 de bază)**: Antrenat pe un corpus masiv și general de text de pe internet.
2.  **LLM Adaptat (GPT-2 după fine-tuning)**: Modelul de bază, antrenat suplimentar pe corpusul specific de poezii.

Diferențele de calitate sunt semnificative și se manifestă în următoarele moduri:

*   **Stil și Structură**:
    *   **Modelul de bază** tinde să genereze text care seamănă mai mult cu proza. Chiar dacă pornește de la un vers, el continuă textul ca pe o propoziție normală, ignorând concepte precum ritmul, rima sau structura în strofe.
    *   **Modelul adaptat** învață stilul din corpusul de poezii. Textul generat de el are o probabilitate mult mai mare să aibă un ritm specific, să folosească un limbaj mai arhaic sau figurat și să se structureze în versuri care au o lungime similară cu cele din datele de antrenare.

*   **Vocabular și Coerență Tematică**:
    *   **Modelul de bază** folosește un vocabular general.
    *   **Modelul adaptat** va alege cuvinte din registrul poetic. Va tinde să continue promptul cu metafore, descrieri artistice și un limbaj specific poeziei.

*   **Concluzie**: Calitatea textului generat de **modelul adaptat este net superioară** pentru sarcina specifică de a scrie poezie. Fine-tuning-ul specializează modelul, transformându-l dintr-un "vorbitor de engleză generalist" într-un "imitator de poet".

### c.2 Ce se întâmplă dacă versurile din prompt sunt în limba engleză?

Acesta este scenariul standard pe care l-am testat. Atât modelul de bază GPT-2, cât și corpusul de poezii sunt în limba engleză.

*   **Rezultat**: Ambele modele vor înțelege promptul și vor genera o continuare în limba engleză. Modelul adaptat va produce o continuare de o calitate poetică superioară, conform celor explicate la punctul c.1.

### c.3 Ce se întâmplă dacă versurile din prompt sunt în limba română?

Aici lucrurile devin interesante. Modelul GPT-2 de bază a "văzut" și texte în limba română în timpul antrenamentului său pe internet, dar engleza este limba sa dominantă.

*   **Modelul de bază (GPT-2)**:
    *   Dacă îi dăm un prompt în română (ex: "Mihai Eminescu a fost un poet"), există o șansă să încerce să continue în română, dar calitatea va fi foarte slabă, textul va fi probabil agramat și incoerent.
    *   Cel mai adesea, va detecta o limbă străină și fie va trece la engleză, fie va genera un text hibrid, bizar (fenomen numit "code-switching").

*   **Modelul adaptat (Fine-Tuned pe poezii în engleză)**:
    *   Acest model a fost forțat să se specializeze **exclusiv** pe poezie în limba engleză. El a "uitat" o mare parte din abilitățile sale generaliste, inclusiv cunoștințele slabe de limba română.
    *   **Rezultat**: Cel mai probabil, va interpreta promptul în română ca pe un șir de caractere fără sens și va genera o continuare în **engleză**, încercând să o facă să sune poetic, dar fără nicio legătură logică cu promptul inițial.

### c.4 Ce se întâmplă dacă versurile din prompt sunt în limba română și corpusul de antrenare este în limba engleză?

Aceasta este exact situația descrisă la punctul c.3 pentru modelul adaptat.

*   **Rezumat**: Modelul nu are nicio "punte" de legătură între limba română (prompt) și limba engleză (cunoștințele sale). Procesul de fine-tuning pe un corpus exclusiv în engleză îl face și mai "ignorant" față de alte limbi. Rezultatul va fi un text generat în engleză, fără legătură cu promptul, deoarece modelul nu înțelege cererea.

### c.5 Cum se poate "personaliza" LLM pentru a genera versuri în stil de pastel (cu accent pe frumusețea naturii)?

Procesul ar fi următorul:

1.  **Crearea unui Corpus Specializat**: În loc să folosim *toate* poeziile din "gutenberg-poetry-corpus", ar trebui să selectam doar poeziile care sunt despre natură. Acest pas este cel mai important și poate fi realizat:
    *   **Manual**: Citind și etichetând un număr de poezii.
    *   **Automat/Semi-automat**: Creând un script care filtrează poeziile pe baza unor cuvinte-cheie relevante pentru natură (ex: "nature", "sun", "moon", "river", "flower", "winter", "forest", "lake", "cloud", etc.).

2.  **Fine-Tuning pe Corpusul de Pasteluri**

3.  **Rezultatul**: Vom obține un nou model, **ultra-specializat**. Când îi vom da un prompt (chiar și unul general), acest model va avea o tendință foarte puternică de a genera versuri care descriu peisaje, anotimpuri și elemente din natură, deoarece aceasta este singura "lume" pe care o cunoaște după acest antrenament specific.

In [21]:
temp_medium = 1.0
print(f"\n[Temp = {temp_medium}]")
print(generate_poem("Afara ninge linistit", max_len, temp=temp_medium, k=50, p=0.95))


[Temp = 1.0]
Afara ninge linistit,

Langit een nei einen 
 Sore mei, nicht 
 I'uld never, I hope 
 Sore ever, 
 Hark ils sind mei, 
 Lags sind


In [22]:
temp_medium = 1.0
print(f"\n[Temp = {temp_medium}]")
print(generate_poem_ft("Afara ninge linistit", max_len, temp=temp_medium, k=50, p=0.95))


[Temp = 1.0]
Afara ninge linistit, en morn.
 Dancin' the moon, or something else, or something more? 
 Nien 
 L'ai, l'ai, laure. 
 
 
         
